# Fase 2 – Preparación, exploración y validación de datos

## Objetivo de la fase

Preparar el conjunto de datos para su análisis mediante un proceso reproducible de carga, exploración, limpieza, transformación y validación, manteniendo trazabilidad sobre cada decisión técnica aplicada.

## Estructura de trabajo

1. Configuración del entorno.
2. Carga del dataset.
3. Exploración inicial.
4. Evaluación de calidad de datos.
5. Limpieza y transformación.
6. Validación del dataset procesado.
7. Exportación de datos preparados.


## 1. Configuración del entorno

Se importan las librerías necesarias para la preparación y exploración de los datos. Se utiliza `Path` para gestionar rutas relativas, permitiendo que el notebook pueda ejecutarse en distintos equipos manteniendo la misma estructura del repositorio.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 2. Carga del dataset

Se define la ruta relativa del archivo original almacenado en `data/raw` y se realiza su carga sin modificar el archivo fuente. El dataset utiliza punto y coma (`;`) como separador de columnas.

In [2]:
ruta_proyecto = Path.cwd().parent
ruta_raw = ruta_proyecto / "data" / "raw" / "dataset_proyecto_vf.csv"

if not ruta_raw.exists():
    raise FileNotFoundError(f"No se encontró el dataset: {ruta_raw}")

df_raw = pd.read_csv(ruta_raw, sep=";")

print("Dataset cargado correctamente.")
print(f"Filas: {df_raw.shape[0]}")
print(f"Columnas: {df_raw.shape[1]}")

Dataset cargado correctamente.
Filas: 5833
Columnas: 17


## 3. Exploración inicial del dataset

Antes de realizar procesos de limpieza o transformación, se examina la estructura original del conjunto de datos. Esta etapa permite conocer las variables disponibles, sus tipos de datos y una muestra de los registros, evitando realizar modificaciones sin una evaluación previa.

In [3]:
print("DIMENSIONES DEL DATASET")
print("-----------------------")
print(f"Filas: {df_raw.shape[0]}")
print(f"Columnas: {df_raw.shape[1]}")

print("\nVARIABLES DISPONIBLES")
print("--------------------")
for i, columna in enumerate(df_raw.columns, start=1):
    print(f"{i}. {columna}")

print("\nPRIMEROS REGISTROS")
display(df_raw.head())

DIMENSIONES DEL DATASET
-----------------------
Filas: 5833
Columnas: 17

VARIABLES DISPONIBLES
--------------------
1. Fecha
2. Dia_Semana
3. SANTA_FE_TG1
4. SANTA_FE_TG2
5. SANTA_FE_TG3
6. PACIFICO_TG1
7. PACIFICO_TG2
8. PACIFICO_TG3
9. LAJA_TG3
10. LAJA_TG4
11. SF _ENERGÍA_TG4
12. SF _ENERGÍA_(NETO)
13. LAJA AP
14. PACIFICO AP
15. SANTA FE AP
16. CMPC BUCALEMU_52EG
17. CMPC BUCALEMU 52ET

PRIMEROS REGISTROS


,Fecha,Dia_Semana,SANTA_FE_TG1,SANTA_FE_TG2,SANTA_FE_TG3,PACIFICO_TG1,PACIFICO_TG2,PACIFICO_TG3,LAJA_TG3,LAJA_TG4,SF _ENERGÍA_TG4,SF _ENERGÍA_(NETO),LAJA AP,PACIFICO AP,SANTA FE AP,CMPC BUCALEMU_52EG,CMPC BUCALEMU 52ET
0,01/01/2026 1:00,jueves,"-19,03","-44,57","-43,82","-29,98","-28,09","-9,54","-26,55","-19,95","-46,80","-40,86","-1,15","-20,05","-9,29","-5,24","-4,92"
1,01/01/2026 2:00,jueves,"-19,02","-44,63","-44,46","-30,04","-27,10","-9,58","-23,46","-33,91","-48,46","-42,36","-11,40","-19,39","-8,01","-5,49","-5,25"
2,01/01/2026 3:00,jueves,"-18,94","-44,61","-44,03","-30,02","-27,35","-9,48","-24,20","-32,18","-49,32","-43,19","-10,28","-19,05","-6,92","-5,44","-5,31"
3,01/01/2026 4:00,jueves,"-19,01","-44,71","-44,49","-32,05","-28,15","-9,56","-26,43","-26,41","-52,38","-45,91","-6,93","-21,55","-7,23","-4,64","-4,59"
4,01/01/2026 5:00,jueves,"-19,06","-44,69","-44,46","-32,10","-30,62","-9,56","-25,09","-29,58","-58,14","-51,60","-8,38","-24,05","-8,07","-4,91","-4,61"


## 4. Evaluación inicial de la calidad de los datos

Se realiza un diagnóstico del dataset original para identificar los tipos de datos, valores faltantes y cantidad de valores únicos por variable. Además, se verifica la existencia de registros completamente duplicados.

Este diagnóstico permitirá fundamentar posteriormente las decisiones de limpieza y transformación.

In [4]:
diagnostico = pd.DataFrame({
    "Tipo_dato": df_raw.dtypes,
    "Valores_nulos": df_raw.isna().sum(),
    "Porcentaje_nulos": (df_raw.isna().mean() * 100).round(2),
    "Valores_unicos": df_raw.nunique()
})

display(diagnostico)

duplicados = df_raw.duplicated().sum()

print(f"\nRegistros duplicados: {duplicados}")

,Tipo_dato,Valores_nulos,Porcentaje_nulos,Valores_unicos
Fecha,str,0,0.0,5832
Dia_Semana,str,0,0.0,7
SANTA_FE_TG1,str,0,0.0,322
SANTA_FE_TG2,str,0,0.0,767
SANTA_FE_TG3,str,0,0.0,1283
PACIFICO_TG1,str,0,0.0,1415
PACIFICO_TG2,str,0,0.0,2453
PACIFICO_TG3,str,0,0.0,378
LAJA_TG3,str,0,0.0,1633
LAJA_TG4,str,0,0.0,2516



Registros duplicados: 0


### Hallazgos del diagnóstico inicial

La evaluación inicial muestra que no se identifican valores nulos en las variables analizadas ni registros completamente duplicados.

Sin embargo, las variables energéticas fueron interpretadas como texto (`str`), lo que impide realizar directamente operaciones estadísticas y matemáticas sobre ellas. Por esta razón, será necesario transformar estas variables a un formato numérico antes de continuar con el análisis.

No se eliminarán registros en esta etapa, ya que no existe evidencia que justifique su eliminación.

## 5. Limpieza y transformación

Antes de modificar los tipos de datos, se inspeccionan algunos valores originales para verificar el formato utilizado en las variables energéticas y definir una estrategia de conversión adecuada.

In [5]:
columnas_medicion = df_raw.columns[2:]

for columna in columnas_medicion:
    print(f"\n{columna}")
    print(df_raw[columna].head().tolist())


SANTA_FE_TG1
['-19,03', '-19,02', '-18,94', '-19,01', '-19,06']

SANTA_FE_TG2
['-44,57', '-44,63', '-44,61', '-44,71', '-44,69']

SANTA_FE_TG3
['-43,82', '-44,46', '-44,03', '-44,49', '-44,46']

PACIFICO_TG1
['-29,98', '-30,04', '-30,02', '-32,05', '-32,10']

PACIFICO_TG2
['-28,09', '-27,10', '-27,35', '-28,15', '-30,62']

PACIFICO_TG3
['-9,54', '-9,58', '-9,48', '-9,56', '-9,56']

LAJA_TG3
['-26,55', '-23,46', '-24,20', '-26,43', '-25,09']

LAJA_TG4
['-19,95', '-33,91', '-32,18', '-26,41', '-29,58']

SF _ENERGÍA_TG4
['-46,80', '-48,46', '-49,32', '-52,38', '-58,14']

SF _ENERGÍA_(NETO)
['-40,86', '-42,36', '-43,19', '-45,91', '-51,60']

LAJA AP
['-1,15', '-11,40', '-10,28', '-6,93', '-8,38']

PACIFICO AP
['-20,05', '-19,39', '-19,05', '-21,55', '-24,05']

SANTA FE AP
['-9,29', '-8,01', '-6,92', '-7,23', '-8,07']

CMPC BUCALEMU_52EG
['-5,24', '-5,49', '-5,44', '-4,64', '-4,91']

CMPC BUCALEMU 52ET
['-4,92', '-5,25', '-5,31', '-4,59', '-4,61']


### 5.1 Creación del dataset de trabajo

Para preservar los datos originales, se crea una copia del dataset sobre la cual se realizarán las transformaciones. De esta forma, `df_raw` se mantiene como referencia del archivo original y `df` se utilizará para el proceso de preparación de datos.

In [6]:
df = df_raw.copy()

print("Copia de trabajo creada correctamente.")
print(f"Dimensiones: {df.shape}")

Copia de trabajo creada correctamente.
Dimensiones: (5833, 17)


### 5.2 Conversión de variables de medición a formato numérico

Las variables de medición fueron cargadas como texto debido al uso de coma como separador decimal. Para permitir operaciones matemáticas y estadísticas, se transforman estas columnas a formato numérico.

La conversión se realiza sobre la copia de trabajo y posteriormente se valida el resultado.

In [15]:
def convertir_a_numerico(serie):
    """
    Convierte una serie con coma decimal a formato numérico.
    """
    serie_texto = serie.astype("string")
    
    serie_convertida = pd.to_numeric(
        serie_texto.str.replace(",", ".", regex=False),
        errors="coerce"
    )
    
    return serie_convertida


columnas_medicion = df.columns[2:]

for columna in columnas_medicion:
    df[columna] = convertir_a_numerico(df[columna])

print("Conversión finalizada.")

Conversión finalizada.


In [13]:
df = df_raw.copy()

print("Dataset de trabajo restaurado desde df_raw.")

Dataset de trabajo restaurado desde df_raw.


In [16]:
print(df.dtypes)

Fecha                     str
Dia_Semana                str
SANTA_FE_TG1          Float64
SANTA_FE_TG2          Float64
SANTA_FE_TG3          Float64
PACIFICO_TG1          Float64
PACIFICO_TG2          Float64
PACIFICO_TG3          Float64
LAJA_TG3              Float64
LAJA_TG4              Float64
SF _ENERGÍA_TG4       Float64
SF _ENERGÍA_(NETO)    Float64
LAJA AP               Float64
PACIFICO AP           Float64
SANTA FE AP           Float64
CMPC BUCALEMU_52EG    Float64
CMPC BUCALEMU 52ET    Float64
dtype: object


### 5.3 Validación de la conversión numérica

Después de convertir las variables de medición a formato numérico, se verifica que la transformación no haya generado valores faltantes como consecuencia de datos que no pudieron ser interpretados correctamente.

In [17]:
nulos_antes = df_raw[columnas_medicion].isna().sum().sum()
nulos_despues = df[columnas_medicion].isna().sum().sum()

print("VALIDACIÓN DE LA CONVERSIÓN")
print("---------------------------")
print(f"Valores nulos antes de convertir: {nulos_antes}")
print(f"Valores nulos después de convertir: {nulos_despues}")

VALIDACIÓN DE LA CONVERSIÓN
---------------------------
Valores nulos antes de convertir: 0
Valores nulos después de convertir: 0


### 5.4 Transformación de la variable Fecha

La variable `Fecha` fue cargada inicialmente como texto. Antes de transformarla a un tipo de fecha y hora, se inspecciona su formato original para definir correctamente el procedimiento de conversión.

In [18]:
print("PRIMERAS FECHAS")
print("----------------")
print(df["Fecha"].head(10).tolist())

print("\nÚLTIMAS FECHAS")
print("--------------")
print(df["Fecha"].tail(10).tolist())


PRIMERAS FECHAS
----------------
['01/01/2026 1:00', '01/01/2026 2:00', '01/01/2026 3:00', '01/01/2026 4:00', '01/01/2026 5:00', '01/01/2026 6:00', '01/01/2026 7:00', '01/01/2026 8:00', '01/01/2026 9:00', '01/01/2026 10:00']

ÚLTIMAS FECHAS
--------------
['31/08/2026 15:00', '31/08/2026 16:00', '31/08/2026 17:00', '31/08/2026 18:00', '31/08/2026 19:00', '31/08/2026 20:00', '31/08/2026 21:00', '31/08/2026 22:00', '31/08/2026 23:00', '01/09/2026 0:00']


### 5.5 Conversión de la variable Fecha a formato temporal

La inspección confirma que la variable `Fecha` utiliza el formato día/mes/año y contiene además la hora de cada registro. Se transforma esta variable desde texto a un tipo de fecha y hora para permitir posteriormente el análisis temporal de los datos.

La conversión se realiza especificando explícitamente el formato original para evitar interpretaciones ambiguas entre día y mes.

In [19]:
df["Fecha"] = pd.to_datetime(
    df["Fecha"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

print("Tipo de dato de Fecha:")
print(df["Fecha"].dtype)

print("\nValores nulos en Fecha:")
print(df["Fecha"].isna().sum())

print("\nPeríodo disponible:")
print(f"Desde: {df['Fecha'].min()}")
print(f"Hasta: {df['Fecha'].max()}")

Tipo de dato de Fecha:
datetime64[us]

Valores nulos en Fecha:
0

Período disponible:
Desde: 2026-01-01 01:00:00
Hasta: 2026-09-01 00:00:00


### 5.6 Verificación de registros temporales duplicados

Aunque no se identificaron filas completamente duplicadas, la variable `Fecha` presenta una cantidad de valores únicos inferior al número total de registros. Por esta razón, se verifica si existen marcas de fecha y hora repetidas antes de decidir cualquier tratamiento sobre estos registros.

In [20]:
fechas_duplicadas = df[df["Fecha"].duplicated(keep=False)]

print("REGISTROS CON FECHA DUPLICADA")
print("-----------------------------")
print(f"Cantidad de registros involucrados: {len(fechas_duplicadas)}")

display(fechas_duplicadas)

REGISTROS CON FECHA DUPLICADA
-----------------------------
Cantidad de registros involucrados: 2


,Fecha,Dia_Semana,SANTA_FE_TG1,SANTA_FE_TG2,SANTA_FE_TG3,PACIFICO_TG1,PACIFICO_TG2,PACIFICO_TG3,LAJA_TG3,LAJA_TG4,SF _ENERGÍA_TG4,SF _ENERGÍA_(NETO),LAJA AP,PACIFICO AP,SANTA FE AP,CMPC BUCALEMU_52EG,CMPC BUCALEMU 52ET
2254,2026-04-04 23:00:00,sábado,-18.79,-43.63,-43.96,-38.01,-22.12,-9.2,-26.99,-20.61,-23.43,-17.26,-1.61,-22.39,-9.04,0.04,0.21
2255,2026-04-04 23:00:00,sábado,-18.91,-43.69,-44.24,-38.02,-21.99,-9.2,-26.18,-22.09,-31.09,-25.09,-4.48,-21.95,-7.02,0.04,0.21


### Hallazgo sobre duplicidad temporal

Se identificaron dos registros asociados a la misma marca temporal (`04/04/2026 23:00`). Sin embargo, las mediciones energéticas de ambos registros presentan valores diferentes, por lo que no corresponden a filas completamente duplicadas.

En consecuencia, estos registros se conservarán durante esta etapa del preprocesamiento. La duplicidad temporal será considerada como una característica que requiere validación antes de atribuirla a un error de calidad de datos, ya que podría estar asociada a condiciones del registro temporal o a particularidades debido al cambio de horaro ocurrido en ese día 


### 5.7 Validación de la continuidad temporal

Dado que el dataset corresponde a registros horarios, se analiza la diferencia temporal entre observaciones consecutivas. El objetivo es identificar posibles horas repetidas, intervalos faltantes o discontinuidades en la serie antes de realizar análisis temporales.

In [21]:
diferencia_horas = df["Fecha"].sort_values().diff()

frecuencia_intervalos = diferencia_horas.value_counts().sort_index()

print("INTERVALOS ENTRE REGISTROS")
print("--------------------------")
display(frecuencia_intervalos)

INTERVALOS ENTRE REGISTROS
--------------------------


Fecha
0 days 00:00:00       1
0 days 01:00:00    5831
Name: count, dtype: int64

### Resultado de la validación temporal

El análisis de continuidad temporal muestra 5.831 intervalos consecutivos de una hora y un intervalo de cero horas.

El intervalo de cero horas corresponde a los dos registros identificados para `04/04/2026 23:00`. Ambos registros presentan mediciones energéticas diferentes, por lo que se mantienen en el dataset y no se consideran filas completamente duplicadas.

No se identificaron intervalos superiores a una hora en la secuencia temporal ordenada. Por lo tanto, no se observan saltos horarios entre las marcas temporales disponibles.

La causa de la duplicidad horaria no se atribuye en esta etapa a un error ni a un fenómeno específico sin realizar previamente una validación adicional.

### 5.8 Validación de la variable Día de la Semana

Se verifica la consistencia entre la variable `Dia_Semana` proporcionada por el dataset y el día de la semana correspondiente a cada fecha. Esta validación permite detectar posibles inconsistencias entre la información temporal y su clasificación semanal.



In [22]:
dias_espanol = {
    0: "lunes",
    1: "martes",
    2: "miércoles",
    3: "jueves",
    4: "viernes",
    5: "sábado",
    6: "domingo"
}

df["Dia_Semana_Calculado"] = df["Fecha"].dt.dayofweek.map(dias_espanol)

inconsistencias_dia = df[
    df["Dia_Semana"].str.lower() != df["Dia_Semana_Calculado"]
]

print("VALIDACIÓN DEL DÍA DE LA SEMANA")
print("--------------------------------")
print(f"Registros evaluados: {len(df)}")
print(f"Inconsistencias encontradas: {len(inconsistencias_dia)}")

VALIDACIÓN DEL DÍA DE LA SEMANA
--------------------------------
Registros evaluados: 5833
Inconsistencias encontradas: 0


### Resultado de la validación del día de la semana

La comparación entre `Dia_Semana` y el día calculado directamente desde la variable `Fecha` no identificó inconsistencias en los 5.833 registros evaluados.

Por lo tanto, la clasificación del día de la semana se considera consistente con la información temporal del dataset.

In [23]:
df = df.drop(columns=["Dia_Semana_Calculado"])

print("Columna auxiliar eliminada.")
print(f"Dimensiones actuales del dataset: {df.shape}")

Columna auxiliar eliminada.
Dimensiones actuales del dataset: (5833, 17)


## 6. Exploración estadística de las variables energéticas

Una vez validados los tipos de datos y la estructura temporal, se realiza una exploración descriptiva de las variables de medición.

El objetivo es conocer su distribución general, rangos y variabilidad, sin clasificar automáticamente valores extremos, negativos o cercanos a cero como errores, debido a que estos pueden responder al comportamiento operacional de las plantas o a las convenciones utilizadas en los registros energéticos.

In [24]:
resumen_estadistico = df[columnas_medicion].describe().T

display(resumen_estadistico)

,count,mean,std,min,25%,50%,75%,max
SANTA_FE_TG1,5833.0,-18.290394,4.355254,-20.43,-19.9,-19.26,-19.0,0.0
SANTA_FE_TG2,5833.0,-39.763343,10.683817,-45.4,-44.09,-42.98,-41.09,0.0
SANTA_FE_TG3,5833.0,-37.868853,10.965914,-44.89,-43.43,-41.93,-38.31,0.0
PACIFICO_TG1,5833.0,-32.364831,7.638994,-38.95,-37.07,-34.76,-30.39,0.0
PACIFICO_TG2,5833.0,-20.068508,8.281639,-36.47,-26.08,-21.74,-14.69,0.0
PACIFICO_TG3,5833.0,-8.561157,1.010127,-11.21,-9.16,-8.68,-8.43,0.01
LAJA_TG3,5833.0,-16.792085,12.168808,-36.45,-26.55,-23.86,0.0,0.0
LAJA_TG4,5833.0,-19.711152,11.554661,-50.37,-28.25,-21.79,-12.68,0.0
SF _ENERGÍA_TG4,5833.0,-34.972177,16.824341,-73.13,-47.7,-37.09,-23.02,0.0
SF _ENERGÍA_(NETO),5833.0,-29.157773,16.209715,-67.16,-41.41,-30.92,-17.16,6.01


### Hallazgos de la exploración estadística

Las 15 variables de medición presentan 5.833 observaciones válidas, lo que confirma que la transformación numérica no produjo pérdida de datos.

Las variables asociadas a los turbogeneradores presentan predominantemente valores negativos de acuerdo con la convención utilizada en el dataset, observándose además valores iguales o cercanos a cero en algunas unidades.

Las variables asociadas a AP presentan rangos que incluyen valores negativos y positivos. Entre ellas, `LAJA AP` presenta valores entre -27,25 y 44,50; `PACIFICO AP`, entre -33,19 y 34,29; y `SANTA FE AP`, entre -17,93 y 63,30.

También se observan diferencias relevantes en la dispersión de las variables. Por ejemplo, `PACIFICO_TG3` presenta menor variabilidad que otras mediciones como `SF _ENERGÍA_TG4`.

En esta etapa, los valores extremos, negativos, positivos o cercanos a cero no serán eliminados automáticamente, debido a que pueden corresponder a condiciones operacionales o a convenciones propias del registro energético. Su interpretación requerirá análisis adicional.

### 6.1 Análisis de valores iguales a cero

Se cuantifica la presencia de valores iguales a cero en las variables de medición. Estos valores no se consideran automáticamente errores, ya que podrían representar condiciones operacionales reales. El objetivo inicial es identificar su frecuencia antes de determinar su tratamiento.

In [25]:
conteo_ceros = (df[columnas_medicion] == 0).sum()

porcentaje_ceros = (
    conteo_ceros / len(df) * 100
).round(2)

resumen_ceros = pd.DataFrame({
    "Cantidad_Ceros": conteo_ceros,
    "Porcentaje_Ceros": porcentaje_ceros
})

display(resumen_ceros)

,Cantidad_Ceros,Porcentaje_Ceros
SANTA_FE_TG1,286,4.9
SANTA_FE_TG2,370,6.34
SANTA_FE_TG3,368,6.31
PACIFICO_TG1,156,2.67
PACIFICO_TG2,154,2.64
PACIFICO_TG3,19,0.33
LAJA_TG3,1725,29.57
LAJA_TG4,949,16.27
SF _ENERGÍA_TG4,398,6.82
SF _ENERGÍA_(NETO),3,0.05


### Hallazgos sobre valores iguales a cero

La presencia de valores iguales a cero varía considerablemente entre las variables de medición.

Las mayores proporciones se observan en `LAJA_TG3`, con 1.725 registros (29,57%), y `LAJA_TG4`, con 949 registros (16,27%). En los turbogeneradores de Santa Fe, la proporción de valores iguales a cero se encuentra aproximadamente entre 4,9% y 6,34%, mientras que en Pacífico las variables TG1 y TG2 presentan aproximadamente 2,6% y TG3 un 0,33%.

Las variables asociadas a AP, Bucalemu y `SF _ENERGÍA_(NETO)` presentan una frecuencia considerablemente menor de valores iguales a cero.

Debido a la frecuencia observada, especialmente en las unidades de Laja, los ceros no serán considerados automáticamente valores atípicos ni datos faltantes. Se mantendrán como observaciones válidas mientras no exista evidencia técnica que justifique su eliminación o modificación.

### 6.2 Distribución temporal de los valores cero

Dado que algunas variables presentan una proporción relevante de registros iguales a cero, se analiza inicialmente su distribución temporal. Se utiliza `LAJA_TG3` como caso de análisis por presentar la mayor frecuencia de ceros.

El propósito es determinar si estos valores aparecen de manera aislada o forman períodos consecutivos, lo que puede aportar información sobre el comportamiento temporal de la variable.

In [26]:
cero_laja_tg3 = df["LAJA_TG3"].eq(0)

grupos = cero_laja_tg3.ne(cero_laja_tg3.shift()).cumsum()

bloques_cero_laja = (
    df[cero_laja_tg3]
    .groupby(grupos[cero_laja_tg3])
    .agg(
        Inicio=("Fecha", "min"),
        Fin=("Fecha", "max"),
        Registros=("Fecha", "size")
    )
    .sort_values("Registros", ascending=False)
)

display(bloques_cero_laja.head(10))

,Inicio,Fin,Registros
LAJA_TG3,,,
9,2026-07-04 13:00:00,2026-08-27 15:00:00,1299
5,2026-05-05 07:00:00,2026-05-20 18:00:00,372
11,2026-08-30 05:00:00,2026-09-01 00:00:00,44
1,2026-01-10 13:00:00,2026-01-10 17:00:00,5
7,2026-05-21 08:00:00,2026-05-21 11:00:00,4
3,2026-04-06 17:00:00,2026-04-06 17:00:00,1


### Hallazgos de la distribución temporal de ceros

El análisis de `LAJA_TG3` muestra que los valores iguales a cero no se distribuyen principalmente como observaciones aisladas, sino que se concentran en períodos consecutivos.

El principal bloque se extiende desde el 04/07/2026 13:00 hasta el 27/08/2026 15:00, con 1.299 registros consecutivos en cero. Un segundo período relevante comprende desde el 05/05/2026 07:00 hasta el 20/05/2026 18:00, con 372 registros.

En conjunto, estos dos períodos concentran 1.671 de los 1.725 valores iguales a cero observados en `LAJA_TG3`, equivalentes aproximadamente al 96,9% de los ceros de esta variable.

Este comportamiento evidencia que los ceros presentan una estructura temporal y no deben ser eliminados o imputados automáticamente. Su causa operacional no puede determinarse únicamente a partir del dataset disponible, por lo que se conservarán para las etapas posteriores de análisis.

### 6.3 Función para identificar períodos consecutivos en cero

Para evitar la repetición de código y facilitar la aplicación del mismo procedimiento a distintas variables, se implementa una función que identifica períodos consecutivos con valores iguales a cero.

La función recibe el dataset y el nombre de una variable, y retorna una tabla con la fecha de inicio, fecha de término y cantidad de registros de cada período detectado.

In [28]:
def identificar_bloques_cero(datos, columna):
    """
    Identifica períodos consecutivos con valores iguales a cero.

    Parámetros:
        datos: DataFrame que contiene los datos.
        columna: nombre de la variable que se desea analizar.

    Retorna:
        DataFrame con inicio, fin y cantidad de registros
        de cada período consecutivo en cero.
    """
    es_cero = datos[columna].eq(0)
    grupos = es_cero.ne(es_cero.shift()).cumsum()

    bloques = (
        datos[es_cero]
        .groupby(grupos[es_cero])
        .agg(
            Inicio=("Fecha", "min"),
            Fin=("Fecha", "max"),
            Registros=("Fecha", "size")
        )
        .sort_values("Registros", ascending=False)
        .reset_index(drop=True)
    )

    return bloques

In [29]:
prueba_bloques = identificar_bloques_cero(df, "LAJA_TG3")

display(prueba_bloques.head(10))

,Inicio,Fin,Registros
0,2026-07-04 13:00:00,2026-08-27 15:00:00,1299
1,2026-05-05 07:00:00,2026-05-20 18:00:00,372
2,2026-08-30 05:00:00,2026-09-01 00:00:00,44
3,2026-01-10 13:00:00,2026-01-10 17:00:00,5
4,2026-05-21 08:00:00,2026-05-21 11:00:00,4
5,2026-04-06 17:00:00,2026-04-06 17:00:00,1


### 6.4 Comparación de períodos en cero entre turbogeneradores

Una vez validada la función de identificación de períodos consecutivos en cero, se aplica el procedimiento a todas las variables correspondientes a turbogeneradores.

El objetivo es comparar la frecuencia y extensión de estos períodos entre las distintas unidades, manteniendo los valores cero como observaciones válidas y sin atribuirles una causa operacional específica.

In [30]:
columnas_tg = [
    "SANTA_FE_TG1",
    "SANTA_FE_TG2",
    "SANTA_FE_TG3",
    "PACIFICO_TG1",
    "PACIFICO_TG2",
    "PACIFICO_TG3",
    "LAJA_TG3",
    "LAJA_TG4",
    "SF _ENERGÍA_TG4"
]

resumen_bloques = []

for columna in columnas_tg:
    bloques = identificar_bloques_cero(df, columna)

    if not bloques.empty:
        resumen_bloques.append({
            "Variable": columna,
            "Total_Ceros": int((df[columna] == 0).sum()),
            "Cantidad_Bloques": len(bloques),
            "Bloque_Mayor": int(bloques["Registros"].max()),
            "Inicio_Bloque_Mayor": bloques.iloc[0]["Inicio"],
            "Fin_Bloque_Mayor": bloques.iloc[0]["Fin"]
        })

resumen_bloques = pd.DataFrame(resumen_bloques)

display(resumen_bloques)

,Variable,Total_Ceros,Cantidad_Bloques,Bloque_Mayor,Inicio_Bloque_Mayor,Fin_Bloque_Mayor
0,SANTA_FE_TG1,286,8,202,2026-08-20 15:00:00,2026-08-29 00:00:00
1,SANTA_FE_TG2,370,2,351,2026-08-17 10:00:00,2026-09-01 00:00:00
2,SANTA_FE_TG3,368,3,350,2026-08-17 11:00:00,2026-09-01 00:00:00
3,PACIFICO_TG1,156,9,51,2026-07-22 18:00:00,2026-07-24 20:00:00
4,PACIFICO_TG2,154,9,58,2026-07-22 19:00:00,2026-07-25 04:00:00
5,PACIFICO_TG3,19,3,14,2026-01-22 21:00:00,2026-01-23 10:00:00
6,LAJA_TG3,1725,6,1299,2026-07-04 13:00:00,2026-08-27 15:00:00
7,LAJA_TG4,949,10,407,2026-04-06 17:00:00,2026-04-23 15:00:00
8,SF _ENERGÍA_TG4,398,7,170,2026-08-21 18:00:00,2026-08-28 19:00:00


### Hallazgos de los períodos consecutivos en cero

La aplicación de la función a los distintos turbogeneradores muestra que la presencia de valores iguales a cero presenta comportamientos diferentes entre las unidades.

`LAJA_TG3` registra el período consecutivo más extenso, con 1.299 registros entre el 04/07/2026 13:00 y el 27/08/2026 15:00. También se identifican períodos prolongados en `SANTA_FE_TG2` y `SANTA_FE_TG3`, con bloques máximos de 351 y 350 registros, respectivamente.

En las unidades de Pacífico, los períodos máximos identificados son considerablemente menores: 51 registros en `PACIFICO_TG1`, 58 en `PACIFICO_TG2` y 14 en `PACIFICO_TG3`.

Los resultados confirman que los valores cero presentan estructuras temporales diferenciadas según la unidad analizada. En consecuencia, estos valores se conservarán en el dataset procesado y no serán eliminados ni imputados automáticamente.

La identificación de estos períodos constituye un hallazgo de calidad y comportamiento de los datos; su interpretación como detenciones, mantenimientos u otras condiciones operacionales requeriría información adicional.

## 7. Validación del dataset procesado

Finalizadas las transformaciones, se realizan controles sobre la estructura y calidad del dataset resultante. Se verifica la conservación del número de registros y variables, la ausencia de valores nulos, el tipo temporal de `Fecha` y el tipo numérico de las variables de medición.

In [31]:
print("VALIDACIÓN FINAL DEL DATASET")
print("----------------------------")

print(f"Filas originales: {df_raw.shape[0]}")
print(f"Filas procesadas: {df.shape[0]}")

print(f"\nColumnas originales: {df_raw.shape[1]}")
print(f"Columnas procesadas: {df.shape[1]}")

print(f"\nValores nulos totales: {df.isna().sum().sum()}")

print(f"\nTipo de Fecha: {df['Fecha'].dtype}")

columnas_no_numericas = [
    columna for columna in columnas_medicion
    if not pd.api.types.is_numeric_dtype(df[columna])
]

print(f"\nVariables de medición no numéricas: {len(columnas_no_numericas)}")

if len(columnas_no_numericas) > 0:
    print(columnas_no_numericas)
else:
    print("Todas las variables de medición son numéricas.")

VALIDACIÓN FINAL DEL DATASET
----------------------------
Filas originales: 5833
Filas procesadas: 5833

Columnas originales: 17
Columnas procesadas: 17

Valores nulos totales: 0

Tipo de Fecha: datetime64[us]

Variables de medición no numéricas: 0
Todas las variables de medición son numéricas.


### 7.1 Pruebas automáticas de validación

Se implementan pruebas automáticas mediante `assert` para verificar que las transformaciones realizadas mantienen las condiciones esperadas del dataset.

Estas pruebas permiten detectar de forma inmediata posibles modificaciones futuras que alteren la estructura, generen valores faltantes o produzcan tipos de datos incompatibles con el análisis.

In [32]:
assert df.shape[0] == df_raw.shape[0], \
    "Error: cambió la cantidad de registros."

assert df.shape[1] == df_raw.shape[1], \
    "Error: cambió la cantidad de variables."

assert df.isna().sum().sum() == 0, \
    "Error: existen valores nulos en el dataset procesado."

assert pd.api.types.is_datetime64_any_dtype(df["Fecha"]), \
    "Error: Fecha no tiene formato datetime."

assert all(
    pd.api.types.is_numeric_dtype(df[columna])
    for columna in columnas_medicion
), "Error: existen variables de medición no numéricas."

print("Todas las pruebas de validación fueron superadas correctamente.")

Todas las pruebas de validación fueron superadas correctamente.


## 8. Exportación del dataset procesado

Una vez completadas las etapas de transformación y validación, se exporta el dataset preparado a la carpeta `data/processed`.

El archivo original ubicado en `data/raw` se mantiene sin modificaciones, permitiendo conservar la trazabilidad entre los datos de origen y el resultado del proceso de preparación.

El dataset procesado conserva los 5.833 registros y las 17 variables originales. Las variables de medición se almacenan en formato numérico y `Fecha` corresponde a una variable temporal validada.

In [34]:
ruta_processed = (
    ruta_proyecto
    / "data"
    / "processed"
    / "dataset_proyecto_procesado.csv"
)

ruta_processed.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(
    ruta_processed,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset procesado guardado correctamente.")
print(f"Ruta: {ruta_processed}")

Dataset procesado guardado correctamente.
Ruta: c:\Users\eliza\Desktop\ESTUDIOS\MAGÍSTER\1 PROGRAMACIÓN PARA LA CIENCIA\Proyecto\data\processed\dataset_proyecto_procesado.csv


### 8.1 Verificación del archivo exportado

Para comprobar la reproducibilidad del proceso, el archivo generado se carga nuevamente desde la carpeta `data/processed`.

Se verifica que el dataset exportado conserve sus dimensiones, no presente valores nulos y mantenga la variable `Fecha` como información temporal interpretable.

In [35]:
df_verificacion = pd.read_csv(
    ruta_processed,
    parse_dates=["Fecha"]
)

print("VERIFICACIÓN DEL ARCHIVO EXPORTADO")
print("----------------------------------")
print(f"Filas: {df_verificacion.shape[0]}")
print(f"Columnas: {df_verificacion.shape[1]}")
print(f"Valores nulos: {df_verificacion.isna().sum().sum()}")
print(f"Tipo de Fecha: {df_verificacion['Fecha'].dtype}")


VERIFICACIÓN DEL ARCHIVO EXPORTADO
----------------------------------
Filas: 5833
Columnas: 17
Valores nulos: 0
Tipo de Fecha: datetime64[us]


In [36]:
assert df_verificacion.shape == df.shape, \
    "Error: las dimensiones cambiaron después de exportar."

assert df_verificacion.isna().sum().sum() == 0, \
    "Error: aparecieron valores nulos después de exportar."

assert pd.api.types.is_datetime64_any_dtype(df_verificacion["Fecha"]), \
    "Error: Fecha no fue recuperada como variable temporal."

print("Archivo procesado exportado y verificado correctamente.")

Archivo procesado exportado y verificado correctamente.
